In [1]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision

In [4]:
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

trainset = datasets.MNIST(root="./Data", train=True, download=True, transform=transform)
testset = datasets.MNIST(root="./Data", train=False, download=True, transform=transform)

trainloader = DataLoader(trainset, batch_size=64, shuffle=True)
testloader = DataLoader(testset, batch_size=64, shuffle=False)

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 9.91M/9.91M [00:06<00:00, 1.49MB/s]
100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 28.9k/28.9k [00:00<00:00, 120kB/s]
100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1.65M/1.65M [00:02<00:00, 715kB/s]
100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 4.54k/4.54k [00:00<00:00, 4.17MB/s]


In [5]:
trainloader = DataLoader(trainset , batch_size = 64 , shuffle = True)
testloader = DataLoader(testset , batch_size = 64)

In [7]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv_layer = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # kernel size = 2, stride = 2

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # kernel size = 2, stride = 2

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # kernel size = 2, stride = 2
     
        )

        self.fc_layers = nn.Sequential(
            nn.Linear(3*3*128, 256),
            nn.ReLU(),

            nn.Linear(256, 10)
            # nn.Linear(784, 128)
        )


    def forward(self, x):
        x = self.conv_layer(x)
        x = x.view(x.size(0), -1) # flattening
        #x = torch.flatten(x, 1)
        x = self.fc_layers(x)

        return x

In [8]:
model = CNN()

In [10]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [13]:
epochs = 10

model.train()
for epoch in range(epochs):
    epoch_training_loss = 0.0

    for images,labels in trainloader:
        optimizer.zero_grad()

        output = model.forward(images)
        loss = criterion(output, labels) # loss fnx
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_training_loss += loss.item()

    print(f"epoch={epoch+1}/{epochs} & loss={epoch_training_loss/len(trainloader)}")

epoch=1/10 & loss=0.047613384920099915
epoch=2/10 & loss=0.03264279485712928
epoch=3/10 & loss=0.025202656236601863
epoch=4/10 & loss=0.019645938685967222
epoch=5/10 & loss=0.015590618113802659
epoch=6/10 & loss=0.01402160204357014
epoch=7/10 & loss=0.011321050677319797
epoch=8/10 & loss=0.009810629535539616
epoch=9/10 & loss=0.00986695644348541
epoch=10/10 & loss=0.007630210397992671


In [14]:
# Evaludate our CNN

correct_labels = 0
total_labels = 0

model.eval()

with torch.no_grad():
    for images, labels in testloader:
        outputs = model.forward(images)
        _, predicted  = torch.max(outputs, 1)

        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)

print(f"accuracy = {correct_labels / total_labels * 100}")

accuracy = 99.05000000000001
